## **FLAN T5 Small**

(Fine-tuned LAnguage Net)

In [2]:
# ! pip install -U transformers sentencepiece

In [14]:
! pip show transformers

Name: transformers
Version: 5.3.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /Users/kanavbansal/Developer/.env_jupyter/lib/python3.13/site-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: sentence-transformers


In [4]:
# ! pip show sentencepiece

### **Step 1: Identify the Model Class**

In [1]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("google/flan-t5-small")

print(config)

T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 1024,
  "d_kv": 64,
  "d_model": 512,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 8,
  "num_heads": 6,
  "num_layers": 8,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "scale_decoder_outputs": false,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {
      "early_stopping": t

### **Step 2: Import the Required Class**

In [2]:
# import the required classes
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

### **Step 3: Load the Tokenizer and Model**

In [3]:
# Load tokenizer
t5_tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path="google/flan-t5-small",
    cache_dir=".models/google/flan-t5-small",
)

print("Vocabulary Size:", t5_tokenizer.vocab_size)

Vocabulary Size: 32100


In [4]:
# Load model
t5_model = AutoModelForSeq2SeqLM.from_pretrained(
    pretrained_model_name_or_path="google/flan-t5-small",
    cache_dir=".models/google/flan-t5-small",
    dtype="auto",    
)

print("Vocab and Embedding Size:", t5_model.encoder.embed_tokens)
print("Vocab and Embedding Size:", t5_model.decoder.embed_tokens)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Vocab and Embedding Size: Embedding(32128, 512)
Vocab and Embedding Size: Embedding(32128, 512)


### **Step 4: Exploring the model architecture**

In [5]:
# You can print the model to take a look at its architecture

t5_model

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

### **Step 5: Generation**

In [6]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# devide = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='mps')

In [12]:
prompt = "translate English to German: How old are you?"

input_ids_map = t5_tokenizer(prompt, return_tensors="pt")

input_ids_map = input_ids_map.to(device)
t5_model = t5_model.to(device)

input_ids_map

{'input_ids': tensor([[13959,  1566,    12,  2968,    10,   571,   625,    33,    25,    58,
             1]], device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}

In [20]:
from transformers import GenerationConfig

generation_config = GenerationConfig(
    max_new_tokens=50,
    temperature=0.7,
    do_sample=True
)

output_ids = t5_model.generate(
    **input_ids_map,
    generation_config=generation_config
)

output_ids

tensor([[    0,  2739,   292,   436,     6,    67,     3,    49,     3,    17,
         11294,    58,     1]], device='mps:0')

In [21]:
t5_tokenizer.decode(output_ids[0], skip_special_tokens=True)

'Wie Sie sind, die er teuer?'

## **Helsinki NLP**

Helsinki-NLP/opus-mt-en-hi

In [1]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("Helsinki-NLP/opus-mt-en-hi")

print(config)

MarianConfig {
  "activation_dropout": 0.0,
  "activation_function": "swish",
  "add_bias_logits": false,
  "add_final_layer_norm": false,
  "architectures": [
    "MarianMTModel"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 512,
  "decoder_attention_heads": 8,
  "decoder_ffn_dim": 2048,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 6,
  "decoder_start_token_id": 61949,
  "decoder_vocab_size": 61950,
  "dropout": 0.1,
  "encoder_attention_heads": 8,
  "encoder_ffn_dim": 2048,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 6,
  "eos_token_id": 0,
  "extra_pos_embeddings": 61950,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "init_std": 0.02,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "max_position_embeddings": 512,
  "model_type": "marian",
  "normalize_before": false,
  "normalize_embedd

In [5]:
# import the required classes
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load tokenizer
opus_tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path="Helsinki-NLP/opus-mt-en-hi",
    cache_dir=".models/Helsinki-NLP/opus-mt-en-hi",
)

print("Vocabulary Size:", opus_tokenizer.vocab_size)

# Load model
opus_model = AutoModelForSeq2SeqLM.from_pretrained(
    pretrained_model_name_or_path="Helsinki-NLP/opus-mt-en-hi",
    cache_dir=".models/Helsinki-NLP/opus-mt-en-hi",
    dtype="auto",    
)

print("Vocab and Embedding Size:", opus_model.model.encoder.embed_tokens)
print("Vocab and Embedding Size:", opus_model.model.decoder.embed_tokens)

/Users/kanavbansal/Developer/.env_jupyter/lib/python3.13/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Vocabulary Size: 61950


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Vocab and Embedding Size: Embedding(61950, 512, padding_idx=61949)
Vocab and Embedding Size: Embedding(61950, 512, padding_idx=61949)


In [6]:
opus_model

MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(61950, 512, padding_idx=61949)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(61950, 512, padding_idx=61949)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [7]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# devide = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='mps')

In [12]:
prompt = "Good Evening, How are you?"

input_ids_map = opus_tokenizer(prompt, return_tensors="pt")

input_ids_map = input_ids_map.to(device)
opus_model = opus_model.to(device)

input_ids_map

{'input_ids': tensor([[ 3398, 13219,     2,   244,    54,    27,    22,     0]],
       device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}

In [13]:
from transformers import GenerationConfig

generation_config = GenerationConfig(
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True
)

output_ids = opus_model.generate(
    **input_ids_map,
    generation_config=generation_config
)

output_ids

Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


tensor([[61949,  3578,  4330,     2,   118,   280,    28,    22,     0]],
       device='mps:0')

In [14]:
opus_tokenizer.decode(output_ids[0], skip_special_tokens=True)

'शुभ शाम, आप कैसे हैं?'

## **Facebook NLLB**

facebook/nllb-200-distilled-600M

### **Understanding Language–Script Codes in Multilingual AI Systems**

When working with multilingual NLP and Generative AI systems, you will often encounter language identifiers such as:

* `hin_Deva`
* `kan_Knda`
* `kas_Arab`
* `kas_Deva`
* `eng_Latn`

These identifiers follow a standard and production-friendly convention that helps AI systems correctly interpret how a language is written.

> **Important:**
> To explore additional supported languages and codes, refer to the official FLORES+ language coverage page hosted on
> **Hugging Face**:
> [https://huggingface.co/datasets/openlanguagedata/flores_plus#language-coverage](https://huggingface.co/datasets/openlanguagedata/flores_plus#language-coverage)

#### **Language_Script Code Format**

All such identifiers follow the same structure:

```text
<language>_<script>
```

* The **language** part specifies the language.
* The **script** part specifies the writing system used for that language.

This distinction is critical in real-world multilingual and GenAI pipelines.

#### **Why Script Matters?**

The `script` tells us **how a particular language is written**.

For example, consider the following two sentences:

```text
मुझे किताब चाहिए
```

```text
mujhe kitaab chahiye
```

Both sentences are in **Hindi**, but:
* the first is written in **Devanagari script** (`Deva`)
* the second is written in **Latin script** (`Latn`)

So, conceptually:
* `hin_Deva` → Hindi written in Devanagari
* `hin_Latn` → Hindi written in Latin characters (romanized form)

#### **How the Model Sees This Difference?**

From the model’s point of view:
* the **character set is completely different**
* the **token IDs are completely different**
* the **subword segmentation is completely different**

Even though humans perceive both examples as the same language, the model treats them as very different inputs.

#### **Common Script Codes**

| Script code | Meaning        | Writing system                                                     |
| ----------- | -------------- | ------------------------------------------------------------------ |
| **Deva**    | Devanagari     | Used for Hindi, Marathi, Sanskrit, Nepali, etc.                    |
| **Knda**    | Kannada script | Used for the Kannada language                                      |
| **Arab**    | Arabic script  | Used for Arabic, Urdu, Kashmiri (one form), Persian, etc.          |
| **Latn**    | Latin script   | Used for English, French, Spanish, German and many other languages |

#### **Codes for Indian Languages**

| Language  | FLORES Code | Script     | Notes                        |
| --------- | ----------- | ---------- | ---------------------------- |
| Hindi     | hin_Deva    | Devanagari | Standard Hindi writing       |
| Bengali   | ben_Beng    | Bengali    | Used for Bangla              |
| Tamil     | tam_Taml    | Tamil      | Native Tamil script          |
| Telugu    | tel_Telu    | Telugu     | Native Telugu script         |
| Marathi   | mar_Deva    | Devanagari | Same script as Hindi         |
| Gujarati  | guj_Gujr    | Gujarati   | Native Gujarati script       |
| Kannada   | kan_Knda    | Kannada    | Native Kannada script        |
| Malayalam | mal_Mlym    | Malayalam  | Native Malayalam script      |
| Punjabi   | pan_Guru    | Gurmukhi   | Punjabi (India)              |
| Odia      | ory_Orya    | Odia       | Earlier called Oriya         |
| Assamese  | asm_Beng    | Bengali    | Assamese uses Bengali script |
| Urdu      | urd_Arab    | Arabic     | Perso-Arabic script          |
| Kashmiri  | kas_Arab    | Arabic     | Common in Kashmir            |
| Kashmiri  | kas_Deva    | Devanagari | Alternate script             |
| Nepali    | nep_Deva    | Devanagari | Same script family as Hindi  |
| Sindhi    | snd_Arab    | Arabic     | Common form                  |
| Sindhi    | snd_Deva    | Devanagari | Alternate form               |
| Sanskrit  | san_Deva    | Devanagari | Classical usage              |


#### **Codes for Popular International Languages**

| Language              | FLORES Code | Script            | Notes                  |
| --------------------- | ----------- | ----------------- | ---------------------- |
| English               | eng_Latn    | Latin             | Global default         |
| Spanish               | spa_Latn    | Latin             |                        |
| French                | fra_Latn    | Latin             |                        |
| German                | deu_Latn    | Latin             |                        |
| Portuguese            | por_Latn    | Latin             |                        |
| Italian               | ita_Latn    | Latin             |                        |
| Russian               | rus_Cyrl    | Cyrillic          |                        |
| Arabic                | arb_Arab    | Arabic            | Modern Standard Arabic |
| Chinese (Simplified)  | zho_Hans    | Han (Simplified)  | Mainland China usage   |
| Chinese (Traditional) | zho_Hant    | Han (Traditional) | Taiwan / HK usage      |
| Japanese              | jpn_Jpan    | Japanese          | Mixed writing system   |
| Korean                | kor_Hang    | Hangul            |                        |
| Turkish               | tur_Latn    | Latin             |                        |
| Indonesian            | ind_Latn    | Latin             |                        |
| Vietnamese            | vie_Latn    | Latin             |                        |
| Thai                  | tha_Thai    | Thai              |                        |
| Hebrew                | heb_Hebr    | Hebrew            |                        |

#### **Key Takeaway**
In modern multilingual AI systems, a language is **not uniquely identified by its language name alone**.
Instead, it is defined as:
```text
(language, script)
```
This is why:
```
kas_Arab  ≠  kas_Deva
hin_Deva  ≠  eng_Latn
```
Even when the spoken language may be the same, the writing system directly affects tokenization, embeddings, retrieval quality, and model behavior.

In [1]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("facebook/nllb-200-distilled-600M")

print(config)

M2M100Config {
  "activation_dropout": 0.0,
  "activation_function": "relu",
  "architectures": [
    "M2M100ForConditionalGeneration"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": 0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0,
  "decoder_layers": 12,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "dtype": "float32",
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0,
  "encoder_layers": 12,
  "eos_token_id": 2,
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "max_position_embeddings": 1024,
  "model_type": "m2m_100",
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "scale_embedding": true,
  "tie_word_embeddings": true,
  "tokenizer_class": "NllbTokenizer",
  "transformers_version": "5.3.0",
  "use_cache": true,
  "vocab_size": 256206
}



In [2]:
# import the required classes
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load tokenizer
nllb_tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path="facebook/nllb-200-distilled-600M",
    cache_dir=".models/facebook/nllb-200-distilled-600M",
    src_lang="eng_Latn",         #### Enter the Source Language_Script
)

print("Vocabulary Size:", nllb_tokenizer.vocab_size)

# Load model
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
    pretrained_model_name_or_path="facebook/nllb-200-distilled-600M",
    cache_dir=".models/facebook/nllb-200-distilled-600M",
    dtype="auto",
    use_safetensors=True     # Forces to load .safetensors
)

print("Vocab and Embedding Size:", nllb_model.model.encoder.embed_tokens)
print("Vocab and Embedding Size:", nllb_model.model.decoder.embed_tokens)

Vocabulary Size: 256204


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Vocab and Embedding Size: M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
Vocab and Embedding Size: M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)


In [3]:
nllb_model

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [4]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# devide = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='mps')

In [5]:
prompt = "Good Evening, How are you?"

input_ids_map = nllb_tokenizer(prompt, return_tensors="pt")

input_ids_map = input_ids_map.to(device)
nllb_model = nllb_model.to(device)

input_ids_map

{'input_ids': tensor([[ 24718, 195655, 248079,  13374,   2442,   1259, 248130,      2,      3]],
       device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}

In [6]:
from transformers import GenerationConfig

generation_config = GenerationConfig(
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True,
    forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("tel_Telu")
)

output_ids = nllb_model.generate(
    **input_ids_map,
    generation_config=generation_config,
    
)

output_ids

Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


tensor([[     2, 256172, 151479, 249541,  14265, 248635,  68159, 248382, 248079,
           7986,  27621,  92831, 248130,      2]], device='mps:0')

In [7]:
nllb_tokenizer.decode(output_ids[0], skip_special_tokens=True)

'శుభ సాయంత్రం, మీరు ఎలా ఉన్నారు?'

## **Facebook BART**

- facebook/bart-large-cnn
- BART (large-sized model), fine-tuned on CNN Daily Mail 

In [4]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("facebook/bart-large-cnn")

print(config)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
BartConfig {
  "_num_labels": 3,
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_final_layer_norm": false,
  "architectures": [
    "BartForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 12,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 12,
  "eos_token_id": 2,
  "force_bos_token_to_be_generated": true,
  "gradient_checkpointing": false,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "init_std": 0.02,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "label2

In [5]:
# import the required classes
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load tokenizer
bart_tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path="facebook/bart-large-cnn",
    cache_dir=".models/facebook/bart-large-cnn",
)

print("Vocabulary Size:", bart_tokenizer.vocab_size)

# Load model
bart_model = AutoModelForSeq2SeqLM.from_pretrained(
    pretrained_model_name_or_path="facebook/bart-large-cnn",
    cache_dir=".models/facebook/bart-large-cnn",
    dtype="auto",
    use_safetensors=True     # Forces to load .safetensors
)

print("Vocab and Embedding Size:", bart_model.model.encoder.embed_tokens)
print("Vocab and Embedding Size:", bart_model.model.decoder.embed_tokens)

Vocabulary Size: 50265


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Vocab and Embedding Size: BartScaledWordEmbedding(50264, 1024, padding_idx=1)
Vocab and Embedding Size: BartScaledWordEmbedding(50264, 1024, padding_idx=1)


In [6]:
bart_model

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [7]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# devide = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='mps')

In [8]:
article = """Machine Learning (ML) and Deep Learning (DL) are two core branches of Artificial Intelligence (AI) that focus on enabling computers to learn from data. While both are used to make predictions and automate decision-making, they differ in how they process data and the complexity of models they use. Understanding the differences between them helps us to choose the right approach for a problem, optimize resources and achieving better results in real-world applications.

Machine Learning
Machine Learning is a branch of Artificial Intelligence that enables computer systems to learn patterns from data and make predictions or decisions without being explicitly programmed by humans.

Works with smaller datasets.
Requires manual feature extraction.
Easier to interpret and implement.
Best for structured data like tables or CSV files.
Types of Machine Learning Algorithms
Different types of ML Algorithms:


Supervised Learning: Here model learns from labelled datasets, where the input and output are clearly defined.
Unsupervised Learning: Here the model identifies patterns or relationships in data without any predefined labels.
Reinforcement Learning: Here the system learns by interacting with an environment and receiving rewards or penalties based on its actions.
Applications:
Spam Emails: Detecting spam emails or fraudulent transactions.
Recommendation Systems: Building recommendation systems for movies, products or content.
Analytics: Predictive analytics in finance, healthcare and marketing.
Deep Learning
Deep Learning uses artificial neural networks with multiple hidden layers that can automatically learn complex patterns from raw data like images, sound, and text. It’s used in applications such as image recognition, natural language processing, and speech recognition.

Learns features automatically from data.
Performs better with large datasets.
Needs GPUs or TPUs for training.
Best for unstructured data such as images, audio or text.
Used in complex applications like self-driving cars or chatbots.
Types of Deep Learning
Deep learning encompasses various architectures, each suited to different types of tasks:

Convolutional Neural Networks: Used for image processing tasks, CNNs are designed to adaptively learn spatial hierarchies of features through convolutional layers.
Recurrent Neural Networks: Ideal for sequential data. RNNs have loops that allow information to persist, effective for tasks like speech recognition and language modeling.
Long Short-Term Memory Networks: A type of RNN that addresses the vanishing gradient problem. They are used for complex sequences including text and speech.
Generative Adversarial Networks: GANs consist of two neural networks that are generator and discriminator that compete against each other, creates synthetic data such as images.
Transformers: Handles long-range dependencies in data. They are the backbone of models like GPT and BERT, used in natural language processing.
Applications
Object Detection: Recognizing objects and faces in images or videos.
Natural Language Processing: Converting spoken language into text and understanding speech commands.
Autonomous Vehicle: Enabling autonomous vehicles to perceive and navigate their environment."""


In [9]:
prompt = f"{article}"

input_ids_map = bart_tokenizer(prompt, return_tensors="pt")

input_ids_map = input_ids_map.to(device)
bart_model = bart_model.to(device)

input_ids_map

{'input_ids': tensor([[    0, 46100, 13807,    36, 10537,    43,     8,  8248, 13807,    36,
         26109,    43,    32,    80,  2731,  9836,     9, 27332,  6558,    36,
         15238,    43,    14,  1056,    15, 10298,  7796,     7,  1532,    31,
           414,     4,   616,   258,    32,   341,     7,   146, 12535,     8,
         31399,   568,    12,  5349,     6,    51, 10356,    11,   141,    51,
           609,   414,     8,     5, 13879,     9,  3092,    51,   304,     4,
         22513,     5,  5550,   227,   106,  2607,   201,     7,  2807,     5,
           235,  1548,    13,    10,   936,     6, 22016,  1915,     8,  9499,
           357,   775,    11,   588,    12,  8331,  2975,     4, 50118, 50118,
         46100, 13807, 50118, 46100, 13807,    16,    10,  6084,     9, 27332,
          6558,    14,  9849,  3034,  1743,     7,  1532,  8117,    31,   414,
             8,   146, 12535,    50,  2390,   396,   145, 16369, 30825,    30,
          5868,     4, 50118, 50118, 2

In [11]:
from transformers import GenerationConfig

generation_config = GenerationConfig(
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True,
)

output_ids = bart_model.generate(
    **input_ids_map,
    generation_config=generation_config,
    
)

output_ids

Both `max_new_tokens` (=100) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


tensor([[    2,     0, 46100, 13807,    36, 10537,    43,     8,  8248, 13807,
            36, 26109,    43,  1056,    15, 10298,  7796,     7,  1532,    31,
           414,     4,  1868,    32,   341,     7,   146, 12535,     8, 31399,
           568,    12,  5349,     4,   125,    51, 10356,    11,   141,    51,
           609,   414,     8,     5, 13879,     9,  3092,    51,   304,     4,
         22513,     5,  5550,   227,   106,  2607,   201,     7,  2807,     5,
           235,  1548,    13,    10,   936,     4,     2]], device='mps:0')

In [13]:
bart_tokenizer.decode(output_ids[0], skip_special_tokens=True)

'Machine Learning (ML) and Deep Learning (DL) focus on enabling computers to learn from data. Both are used to make predictions and automate decision-making. But they differ in how they process data and the complexity of models they use. Understanding the differences between them helps us to choose the right approach for a problem.'